# 🎓 The Dropout Signal — ML, Fairness & Explainability
## Models → Fairness Audit → SHAP → Gold Table | HackBricks 2026

**This notebook:**
1. **Two Models** — Logistic Regression (baseline) + XGBoost (tree)
2. **MLflow** — Log params, metrics, artifacts. Register champion.
3. **Fairness Audit** — Demographic parity + equal opportunity across gender & socioeconomic groups
4. **SHAP** — Per-student top 3 risk factors
5. **Gold Table** — Final output: student_id, risk_score, tier, top 3 factors, action

**Prerequisite:** Run `dropout_pipeline.ipynb` first to create Bronze & Silver tables.

In [ ]:
%pip install xgboost shap "numpy<2" --quiet
dbutils.library.restartPython()

In [ ]:
# ============================================================================
#  Cell 1: Imports & Config
# ============================================================================
import sys, io, logging, warnings
warnings.filterwarnings("ignore")

# Suppress noisy PySpark/GRPC loggers
for _ln in ["pyspark", "py4j", "mlflow", "grpc", "pyspark.sql.connect.client.core"]:
    logging.getLogger(_ln).setLevel(logging.CRITICAL)

# Import MLflow with stderr suppressed (protobuf deprecation warning)
_stderr = sys.stderr
sys.stderr = io.StringIO()
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models.signature import infer_signature
sys.stderr = _stderr

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap

# ── Config ───────────────────────────────────────────────────────────
CATALOG   = "workspace"
SCHEMA    = "default"
SILVER_TABLE   = f"{CATALOG}.{SCHEMA}.dropout_silver"
GOLD_TABLE     = f"{CATALOG}.{SCHEMA}.dropout_gold"
FAIRNESS_TABLE = f"{CATALOG}.{SCHEMA}.dropout_fairness_audit"

EXPERIMENT_NAME = "/dropout-signal-experiment"
RANDOM_STATE = 42

# Suppress stderr during MLflow experiment setup (GRPC noise)
_stderr = sys.stderr
try:
    sys.stderr = io.StringIO()
    mlflow.set_experiment(EXPERIMENT_NAME)
    sys.stderr = _stderr
    print(f"✅ MLflow experiment: {EXPERIMENT_NAME}")
except Exception as e:
    sys.stderr = _stderr
    print(f"⚠️  MLflow experiment note (non-blocking): {e}")
    print(f"   Using default experiment instead.")

print(f"📊 Silver table: {SILVER_TABLE}")

In [ ]:
# ============================================================================
#  Cell 2: Load Silver table + Prepare Features & Target
# ============================================================================

# Load Silver table
silver_df = spark.table(SILVER_TABLE).toPandas()
print(f"📊 Loaded {len(silver_df):,} rows from Silver table")

# ── Identify feature columns ─────────────────────────────────────────
# Drop non-feature columns
drop_cols = ["target", "is_dropout", "_ingestion_ts", "_source_file"]
meta_cols = [c for c in silver_df.columns if c.startswith("_")]
drop_cols.extend(meta_cols)

# Keep only numeric columns as features
feature_cols = [
    c for c in silver_df.columns
    if c not in drop_cols
    and silver_df[c].dtype in ["int64", "float64", "int32", "float32"]
]

# Store protected attributes BEFORE dropping them from features
# (used for fairness audit later)
gender_col = [c for c in silver_df.columns if "gender" in c.lower()]
gender_col = gender_col[0] if gender_col else None

# Build X, y
X = silver_df[feature_cols].copy()
y = silver_df["is_dropout"].copy()

# Fill any remaining NaNs
X = X.fillna(0)

print(f"\n📋 Features: {len(feature_cols)} columns")
print(f"🎯 Target: is_dropout (1=Dropout, 0=Enrolled/Graduate)")
print(f"   Dropout rate: {y.mean():.1%} ({y.sum()} / {len(y)})")
print(f"\n🔐 Protected attribute for fairness: '{gender_col}'")

# ── Train/Test Split ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\n✂️  Train: {len(X_train):,} | Test: {len(X_test):,}")

---
## 📊 Model A: Logistic Regression (Baseline)
Interpretable linear model. Regularized with L2 penalty. Serves as the benchmark for our tree model.

In [ ]:
# ============================================================================
#  Cell 3: Model A — Logistic Regression + MLflow Logging
# ============================================================================

# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    penalty="l2",
    solver="lbfgs",
    random_state=RANDOM_STATE
)
lr_model.fit(X_train_scaled, y_train)

# Predict
y_pred_lr  = lr_model.predict(X_test_scaled)
y_prob_lr  = lr_model.predict_proba(X_test_scaled)[:, 1]

# Metrics
lr_metrics = {
    "accuracy":  accuracy_score(y_test, y_pred_lr),
    "precision": precision_score(y_test, y_pred_lr, zero_division=0),
    "recall":    recall_score(y_test, y_pred_lr, zero_division=0),
    "f1_score":  f1_score(y_test, y_pred_lr, zero_division=0),
    "auc_roc":   roc_auc_score(y_test, y_prob_lr),
}

# Log to MLflow
lr_run_id = None
try:
    _stderr = sys.stderr
    sys.stderr = io.StringIO()
    with mlflow.start_run(run_name="LogisticRegression_Baseline") as run_lr:
        mlflow.log_params({"model": "LogisticRegression", "C": 1.0, "penalty": "l2", "max_iter": 1000})
        mlflow.log_metrics(lr_metrics)
        signature = infer_signature(X_train_scaled, y_pred_lr)
        mlflow.sklearn.log_model(lr_model, "model", signature=signature)
        lr_run_id = run_lr.info.run_id
    sys.stderr = _stderr
    print(f"  ✅ MLflow logged: {lr_run_id}")
except Exception as e:
    sys.stderr = _stderr
    print(f"  ⚠️  MLflow logging skipped (non-blocking)")
    lr_run_id = "mlflow-unavailable"

print("=" * 60)
print("  MODEL A: Logistic Regression (Baseline)")
print("=" * 60)

print(f"\n  MLflow Run ID: {lr_run_id}")
for k, v in lr_metrics.items():
    print(f"  {k:<12}: {v:.4f}")

---
## 🌲 Model B: XGBoost (Tree Model)
Gradient boosted trees — handles non-linear relationships and feature interactions. Our expected champion.

In [ ]:
# ============================================================================
#  Cell 4: Model B — XGBoost + MLflow Logging
# ============================================================================

# Train (XGBoost works on raw features, no scaling needed)
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    use_label_encoder=False,
)
xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Metrics
xgb_metrics = {
    "accuracy":  accuracy_score(y_test, y_pred_xgb),
    "precision": precision_score(y_test, y_pred_xgb, zero_division=0),
    "recall":    recall_score(y_test, y_pred_xgb, zero_division=0),
    "f1_score":  f1_score(y_test, y_pred_xgb, zero_division=0),
    "auc_roc":   roc_auc_score(y_test, y_prob_xgb),
}

# Log to MLflow
xgb_run_id = None
try:
    _stderr = sys.stderr
    sys.stderr = io.StringIO()
    with mlflow.start_run(run_name="XGBoost_TreeModel") as run_xgb:
        mlflow.log_params({
            "model": "XGBoost",
            "n_estimators": 200,
            "max_depth": 6,
            "learning_rate": 0.1,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
        })
        mlflow.log_metrics(xgb_metrics)
        signature = infer_signature(X_train, y_pred_xgb)
        mlflow.xgboost.log_model(xgb_model, "model", signature=signature)
        xgb_run_id = run_xgb.info.run_id
    sys.stderr = _stderr
    print(f"  ✅ MLflow logged: {xgb_run_id}")
except Exception as e:
    sys.stderr = _stderr
    print(f"  ⚠️  MLflow logging skipped (non-blocking)")
    xgb_run_id = "mlflow-unavailable"

print("=" * 60)
print("  MODEL B: XGBoost (Tree Model)")
print("=" * 60)

print(f"\n  MLflow Run ID: {xgb_run_id}")
for k, v in xgb_metrics.items():
    print(f"  {k:<12}: {v:.4f}")

In [ ]:
# ============================================================================
#  Cell 5: Model Comparison + Register Champion
# ============================================================================

print("=" * 60)
print("  MODEL COMPARISON")
print("=" * 60)
print(f"  {'Metric':<14} {'Logistic Reg':>14} {'XGBoost':>14}  {'Winner':>10}")
print("  " + "─" * 56)

results = {}
for metric in ["accuracy", "precision", "recall", "f1_score", "auc_roc"]:
    lr_val  = lr_metrics[metric]
    xgb_val = xgb_metrics[metric]
    winner  = "XGBoost" if xgb_val >= lr_val else "LogReg"
    print(f"  {metric:<14} {lr_val:>14.4f} {xgb_val:>14.4f}  {'← ' + winner:>10}")
    results[metric] = winner

# Determine champion based on F1 score (balances precision & recall)
if xgb_metrics["f1_score"] >= lr_metrics["f1_score"]:
    champion_name = "XGBoost"
    champion_model = xgb_model
    champion_run_id = xgb_run_id
    champion_metrics = xgb_metrics
    y_pred_best = y_pred_xgb
    y_prob_best = y_prob_xgb
    uses_scaler = False
else:
    champion_name = "LogisticRegression"
    champion_model = lr_model
    champion_run_id = lr_run_id
    champion_metrics = lr_metrics
    y_pred_best = y_pred_lr
    y_prob_best = y_prob_lr
    uses_scaler = True

print(f"\n  🏆 CHAMPION: {champion_name} (F1: {champion_metrics['f1_score']:.4f})")

# Register champion in Model Registry
model_uri = f"runs:/{champion_run_id}/model"
model_name = f"{CATALOG}.{SCHEMA}.dropout_champion_model"

try:
    _stderr = sys.stderr
    sys.stderr = io.StringIO()
    reg_result = mlflow.register_model(model_uri, model_name)
    sys.stderr = _stderr
    print(f"  ✅ Registered as: {model_name} (version {reg_result.version})")

except Exception as e:
    sys.stderr = _stderr
    print(f"  ⚠️  Model registration note (non-blocking)")
    print(f"  ℹ️  Model logged in MLflow run: {champion_run_id}")

---
## ⚖️ FAIRNESS AUDIT (25% of Evaluation Score)

We compute **Demographic Parity** and **Equal Opportunity** across:
- **Gender** (Male vs Female)
- **Socioeconomic group** (scholarship holder + tuition payment status)

> *"Document any disparity found — do not hide it."* — Problem Statement

In [ ]:
# ============================================================================
#  Cell 6: FAIRNESS AUDIT — Full Bias Analysis
# ============================================================================

# ── Prepare test set with protected attributes ───────────────────────
test_indices = X_test.index
test_audit = silver_df.loc[test_indices].copy()
test_audit["y_true"] = y_test.values
test_audit["y_pred"] = y_pred_best
test_audit["risk_score"] = y_prob_best

# ── Identify protected attribute columns ─────────────────────────────
gender_col = [c for c in test_audit.columns if "gender" in c.lower()]
scholarship_col = [c for c in test_audit.columns if "scholarship" in c.lower()]
tuition_col = [c for c in test_audit.columns if "tuition" in c.lower()]

gender_col = gender_col[0] if gender_col else None
scholarship_col = scholarship_col[0] if scholarship_col else None
tuition_col = tuition_col[0] if tuition_col else None

# Create socioeconomic group: "disadvantaged" vs "advantaged"
if scholarship_col and tuition_col:
    test_audit["socioeconomic_group"] = np.where(
        (test_audit[scholarship_col] == 1) | (test_audit[tuition_col] == 0),
        "disadvantaged",
        "advantaged"
    )
elif scholarship_col:
    test_audit["socioeconomic_group"] = np.where(
        test_audit[scholarship_col] == 1, "scholarship_holder", "no_scholarship"
    )

# ── Fairness metrics functions ───────────────────────────────────────
def compute_fairness_metrics(df, group_col, group_a_val, group_b_val, y_true_col="y_true", y_pred_col="y_pred"):
    """Compute demographic parity & equal opportunity between two groups."""
    group_a = df[df[group_col] == group_a_val]
    group_b = df[df[group_col] == group_b_val]

    # Demographic Parity: P(predicted=1 | Group)
    rate_a = group_a[y_pred_col].mean() if len(group_a) > 0 else 0
    rate_b = group_b[y_pred_col].mean() if len(group_b) > 0 else 0
    dp_ratio = rate_a / rate_b if rate_b > 0 else float("inf")

    # Equal Opportunity: TPR per group (recall for actual positives)
    tp_a = group_a[(group_a[y_true_col] == 1) & (group_a[y_pred_col] == 1)]
    fn_a = group_a[(group_a[y_true_col] == 1) & (group_a[y_pred_col] == 0)]
    tpr_a = len(tp_a) / (len(tp_a) + len(fn_a)) if (len(tp_a) + len(fn_a)) > 0 else 0

    tp_b = group_b[(group_b[y_true_col] == 1) & (group_b[y_pred_col] == 1)]
    fn_b = group_b[(group_b[y_true_col] == 1) & (group_b[y_pred_col] == 0)]
    tpr_b = len(tp_b) / (len(tp_b) + len(fn_b)) if (len(tp_b) + len(fn_b)) > 0 else 0

    eo_ratio = tpr_a / tpr_b if tpr_b > 0 else float("inf")

    # Predictive Parity: Precision per group
    prec_a = len(tp_a) / (len(tp_a) + len(group_a[(group_a[y_true_col] == 0) & (group_a[y_pred_col] == 1)])) if len(group_a[group_a[y_pred_col] == 1]) > 0 else 0
    prec_b = len(tp_b) / (len(tp_b) + len(group_b[(group_b[y_true_col] == 0) & (group_b[y_pred_col] == 1)])) if len(group_b[group_b[y_pred_col] == 1]) > 0 else 0
    pp_ratio = prec_a / prec_b if prec_b > 0 else float("inf")

    return {
        "group_a": str(group_a_val),
        "group_b": str(group_b_val),
        "group_a_count": len(group_a),
        "group_b_count": len(group_b),
        "flag_rate_a": round(rate_a, 4),
        "flag_rate_b": round(rate_b, 4),
        "demographic_parity_ratio": round(dp_ratio, 4),
        "dp_is_fair": "YES" if 0.8 <= dp_ratio <= 1.25 else "NO",
        "tpr_a": round(tpr_a, 4),
        "tpr_b": round(tpr_b, 4),
        "equal_opportunity_ratio": round(eo_ratio, 4),
        "eo_is_fair": "YES" if 0.8 <= eo_ratio <= 1.25 else "NO",
        "precision_a": round(prec_a, 4),
        "precision_b": round(prec_b, 4),
        "predictive_parity_ratio": round(pp_ratio, 4),
        "pp_is_fair": "YES" if 0.8 <= pp_ratio <= 1.25 else "NO",
    }

# ── Compute fairness across protected attributes ─────────────────────
fairness_results = []

# 1. Gender audit
if gender_col:
    gender_values = sorted(test_audit[gender_col].unique())
    if len(gender_values) >= 2:
        gender_metrics = compute_fairness_metrics(
            test_audit, gender_col, gender_values[0], gender_values[1]
        )
        gender_metrics["protected_attribute"] = "gender"
        gender_metrics["model"] = champion_name
        fairness_results.append(gender_metrics)
        print(f"✅ Gender audit computed: {gender_col} = {gender_values}")

# 2. Socioeconomic audit
if "socioeconomic_group" in test_audit.columns:
    se_metrics = compute_fairness_metrics(
        test_audit, "socioeconomic_group", "disadvantaged", "advantaged"
    )
    se_metrics["protected_attribute"] = "socioeconomic"
    se_metrics["model"] = champion_name
    fairness_results.append(se_metrics)
    print(f"✅ Socioeconomic audit computed")

# ── Save fairness audit to Delta table ────────────────────────────────
if fairness_results:
    fairness_pdf = pd.DataFrame(fairness_results)
    fairness_sdf = spark.createDataFrame(fairness_pdf)
    fairness_sdf.write.format("delta").mode("overwrite").saveAsTable(FAIRNESS_TABLE)
    print(f"\n✅ Fairness audit saved to: {FAIRNESS_TABLE}")

In [ ]:
# ============================================================================
#  Cell 7: FAIRNESS — Display Results + Analysis
# ============================================================================

print("=" * 70)
print("  ⚖️  FAIRNESS AUDIT REPORT")
print("  Model:", champion_name)
print("=" * 70)

for result in fairness_results:
    attr = result["protected_attribute"].upper()
    print(f"\n  ── {attr}: {result['group_a']} vs {result['group_b']} ──")
    print(f"     Group sizes: {result['group_a_count']} vs {result['group_b_count']}")
    print()

    # Demographic Parity
    dp = result["demographic_parity_ratio"]
    dp_fair = result["dp_is_fair"]
    dp_icon = "✅" if dp_fair == "YES" else "⚠️"
    print(f"     {dp_icon} Demographic Parity Ratio:   {dp:.4f}  [{dp_fair}]")
    print(f"        Flag rate ({result['group_a']}): {result['flag_rate_a']:.1%}")
    print(f"        Flag rate ({result['group_b']}): {result['flag_rate_b']:.1%}")
    print(f"        Fair range: 0.80 – 1.25")
    print()

    # Equal Opportunity
    eo = result["equal_opportunity_ratio"]
    eo_fair = result["eo_is_fair"]
    eo_icon = "✅" if eo_fair == "YES" else "⚠️"
    print(f"     {eo_icon} Equal Opportunity Ratio:    {eo:.4f}  [{eo_fair}]")
    print(f"        TPR ({result['group_a']}): {result['tpr_a']:.1%}")
    print(f"        TPR ({result['group_b']}): {result['tpr_b']:.1%}")
    print(f"        Fair range: 0.80 – 1.25")
    print()

    # Predictive Parity
    pp = result["predictive_parity_ratio"]
    pp_fair = result["pp_is_fair"]
    pp_icon = "✅" if pp_fair == "YES" else "⚠️"
    print(f"     {pp_icon} Predictive Parity Ratio:   {pp:.4f}  [{pp_fair}]")
    print(f"        Precision ({result['group_a']}): {result['precision_a']:.1%}")
    print(f"        Precision ({result['group_b']}): {result['precision_b']:.1%}")
    print(f"        Fair range: 0.80 – 1.25")

    # Documentation of any disparity
    disparities = []
    if dp_fair == "NO":
        disparities.append(f"Demographic Parity ({dp:.2f})")
    if eo_fair == "NO":
        disparities.append(f"Equal Opportunity ({eo:.2f})")
    if pp_fair == "NO":
        disparities.append(f"Predictive Parity ({pp:.2f})")

    if disparities:
        print(f"\n     📋 DISPARITY DOCUMENTED: {', '.join(disparities)}")
        print(f"        Mitigation options:")
        print(f"        • Adjust classification threshold per group")
        print(f"        • Re-weight training samples to balance group representation")
        print(f"        • Remove direct socioeconomic features and rely on academic signals")
    else:
        print(f"\n     ✅ No significant disparities found for {attr}.")

print("\n" + "=" * 70)
print(f"  Audit saved to Delta table: {FAIRNESS_TABLE}")
print("=" * 70)

# Display the fairness table
display(spark.table(FAIRNESS_TABLE))

---
## 🔍 SHAP Explainability (15% of Evaluation Score)

Two outputs:
1. **Global** — Which features matter most overall (SHAP summary plot)
2. **Per-student** — Top 3 risk factors for each flagged student

In [ ]:
# ============================================================================
#  Cell 8: SHAP — Global Feature Importance
# ============================================================================
import builtins

# Force all values to float (Spark sometimes returns bracketed strings)
def clean_numeric_df(df):
    """Strip brackets and convert all values to float64."""
    out = df.copy()
    for col in out.columns:
        out[col] = pd.to_numeric(
            out[col].astype(str).str.strip('[]'), errors='coerce'
        )
    return out.fillna(0).astype(np.float64)

X_test_clean = clean_numeric_df(X_test)
X_train_clean = clean_numeric_df(X_train)

# Monkey-patch float() to handle XGBoost's bracketed base_score like '[3.21E-1]'
_orig_float = builtins.float
def _safe_float(x):
    if isinstance(x, str) and x.startswith("[") and x.endswith("]"):
        return _orig_float(x.strip("[]"))
    return _orig_float(x)

if champion_name == "XGBoost":
    builtins.float = _safe_float          # patch ON
    explainer = shap.TreeExplainer(champion_model)
    builtins.float = _orig_float          # patch OFF
    shap_values = explainer.shap_values(X_test_clean.values)
    X_shap = X_test_clean
else:
    # For Logistic Regression, use a sample to speed up KernelExplainer
    sample_size = min(200, len(X_test_scaled))
    background = shap.kmeans(X_train_scaled, 10)
    explainer = shap.KernelExplainer(champion_model.predict_proba, background)
    shap_values = explainer.shap_values(X_test_scaled[:sample_size])
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # class 1 (dropout)
    X_shap = pd.DataFrame(X_test_scaled[:sample_size], columns=feature_cols)

# ── Global Summary Plot ──────────────────────────────────────────────
print("📊 SHAP Global Feature Importance (Top 15)")
print("   Which features drive dropout predictions overall?\n")

shap.summary_plot(shap_values, X_shap, max_display=15, show=False)
import matplotlib.pyplot as plt
plt.title("SHAP Feature Importance — The Dropout Signal", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
#  Cell 9: SHAP — Per-Student Top 3 Risk Factors
# ============================================================================

def get_top3_shap_factors(shap_vals, feature_names, student_idx):
    """Get top 3 SHAP features driving a student's risk score."""
    student_shap = shap_vals[student_idx]
    feature_impact = list(zip(feature_names, student_shap))
    # Sort by absolute SHAP value (descending)
    feature_impact.sort(key=lambda x: abs(x[1]), reverse=True)
    top3 = feature_impact[:3]
    return [(name, round(float(val), 4)) for name, val in top3]

# ── Build per-student explanations for high-risk students ────────────
high_risk_mask = y_prob_best >= 0.5
high_risk_indices = np.where(high_risk_mask)[0]

print(f"🔍 High-risk students (risk ≥ 0.50): {len(high_risk_indices)}")
print(f"   Generating per-student SHAP explanations...\n")

shap_explanations = []
for idx in high_risk_indices:
    top3 = get_top3_shap_factors(shap_values, feature_cols, idx)
    student_data = X_test.iloc[idx]
    original_idx = X_test.index[idx]

    shap_explanations.append({
        "student_idx": int(original_idx),
        "risk_score": round(float(y_prob_best[idx]), 4),
        "predicted_dropout": int(y_pred_best[idx]),
        "actual_dropout": int(y_test.iloc[idx]),
        "factor_1_name": top3[0][0],
        "factor_1_shap": top3[0][1],
        "factor_1_value": round(float(student_data[top3[0][0]]), 4),
        "factor_2_name": top3[1][0],
        "factor_2_shap": top3[1][1],
        "factor_2_value": round(float(student_data[top3[1][0]]), 4),
        "factor_3_name": top3[2][0],
        "factor_3_shap": top3[2][1],
        "factor_3_value": round(float(student_data[top3[2][0]]), 4),
    })

# Show top 5 highest-risk students
print("=" * 70)
print("  TOP 5 HIGHEST-RISK STUDENTS — Per-Student SHAP Explanation")
print("=" * 70)

sorted_explanations = sorted(shap_explanations, key=lambda x: x["risk_score"], reverse=True)
for i, exp in enumerate(sorted_explanations[:5]):
    correct = "✓" if exp["predicted_dropout"] == exp["actual_dropout"] else "✗"
    print(f"\n  Student #{exp['student_idx']} — Risk: {exp['risk_score']:.1%}  [Actual: {'Dropout' if exp['actual_dropout'] else 'Stayed'}] {correct}")
    print(f"  ┌─ #1: {exp['factor_1_name']:<40} SHAP: {exp['factor_1_shap']:+.4f}  Value: {exp['factor_1_value']}")
    print(f"  ├─ #2: {exp['factor_2_name']:<40} SHAP: {exp['factor_2_shap']:+.4f}  Value: {exp['factor_2_value']}")
    print(f"  └─ #3: {exp['factor_3_name']:<40} SHAP: {exp['factor_3_shap']:+.4f}  Value: {exp['factor_3_value']}")

In [ ]:
# ============================================================================
#  Cell 10: SHAP — Waterfall Plot for Highest-Risk Student (Visual)
# ============================================================================

# Show SHAP waterfall for the single highest-risk student
if len(sorted_explanations) > 0:
    top_student_idx_in_test = np.argmax(y_prob_best)

    print(f"📊 SHAP Waterfall — Student #{X_test.index[top_student_idx_in_test]} (Risk: {y_prob_best[top_student_idx_in_test]:.1%})")
    print(f"   Shows exactly how each feature pushes the prediction toward dropout.\n")

    shap_explanation = shap.Explanation(
        values=shap_values[top_student_idx_in_test],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value[1],
        data=X_shap.iloc[top_student_idx_in_test].values,
        feature_names=feature_cols,
    )

    shap.waterfall_plot(shap_explanation, max_display=10, show=False)
    plt.title(f"SHAP Waterfall — Student #{X_test.index[top_student_idx_in_test]}", fontsize=12)
    plt.tight_layout()
    plt.show()

---
## 🥇 GOLD TABLE — Final Deliverable

The Gold table is the university's **action list**: every at-risk student with their risk score, top 3 risk factors (from SHAP), and recommended intervention tier.

In [ ]:
# ============================================================================
#  Cell 11: GOLD TABLE — Score ALL Students + Create Final Output
# ============================================================================

# ── Score the ENTIRE dataset (not just test set) ─────────────────────
all_X = clean_numeric_df(silver_df[feature_cols])

if uses_scaler:
    all_X_transformed = scaler.transform(all_X)
    all_probs = champion_model.predict_proba(all_X_transformed)[:, 1]
    all_preds = champion_model.predict(all_X_transformed)
else:
    all_probs = champion_model.predict_proba(all_X.values)[:, 1]
    all_preds = champion_model.predict(all_X.values)

# ── Compute SHAP for all students ────────────────────────────────────
print("⏳ Computing SHAP values for all students (this may take a moment)...")
if champion_name == "XGBoost":
    all_shap_values = explainer.shap_values(all_X.values)
else:
    all_shap_values = explainer.shap_values(all_X_transformed)
    if isinstance(all_shap_values, list):
        all_shap_values = all_shap_values[1]

# ── Build Gold table rows ────────────────────────────────────────────
gold_rows = []
for i in range(len(silver_df)):
    risk_score = float(all_probs[i])

    # Intervention tier
    if risk_score >= 0.7:
        tier = "HIGH"
        action = "Immediate 1-on-1 counseling + financial aid review"
    elif risk_score >= 0.4:
        tier = "MEDIUM"
        action = "Email alert + advisor check-in within 1 week"
    else:
        tier = "LOW"
        action = "Monitor only — no immediate action"

    # Top 3 SHAP factors
    top3 = get_top3_shap_factors(all_shap_values, feature_cols, i)

    gold_rows.append({
        "student_id": i + 1,
        "risk_score": round(risk_score, 4),
        "predicted_dropout": int(all_preds[i]),
        "actual_outcome": silver_df.iloc[i].get("target", "Unknown"),
        "risk_tier": tier,
        "top_factor_1": f"{top3[0][0]} ({top3[0][1]:+.3f})",
        "top_factor_2": f"{top3[1][0]} ({top3[1][1]:+.3f})",
        "top_factor_3": f"{top3[2][0]} ({top3[2][1]:+.3f})",
        "recommended_action": action,
    })

# ── Write Gold Delta table ───────────────────────────────────────────
gold_pdf = pd.DataFrame(gold_rows)
gold_sdf = spark.createDataFrame(gold_pdf)
gold_sdf.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)

print(f"✅ Gold table created: {GOLD_TABLE}")
print(f"   Total students: {len(gold_rows):,}")
print(f"   HIGH risk:   {sum(1 for r in gold_rows if r['risk_tier'] == 'HIGH'):,}")
print(f"   MEDIUM risk: {sum(1 for r in gold_rows if r['risk_tier'] == 'MEDIUM'):,}")
print(f"   LOW risk:    {sum(1 for r in gold_rows if r['risk_tier'] == 'LOW'):,}")

In [ ]:
# ============================================================================
#  Cell 12: GOLD TABLE — Display Sample Output
# ============================================================================

from pyspark.sql import functions as F

print("=" * 70)
print("  🥇 GOLD TABLE — Student Risk Assessment Output")
print("=" * 70)
print()

# Show high-risk students
display(
    spark.table(GOLD_TABLE)
    .filter("risk_tier = 'HIGH'")
    .orderBy(F.desc("risk_score"))
    .limit(10)
)

# Summary
print("\n📊 Risk Tier Distribution:")
display(
    spark.table(GOLD_TABLE)
    .groupBy("risk_tier")
    .agg(
        F.count("*").alias("student_count"),
        F.round(F.avg("risk_score"), 3).alias("avg_risk_score"),
    )
    .orderBy(F.desc("avg_risk_score"))
)

---
## ✅ ML Notebook Complete

| Step | Deliverable | Status |
|------|-------------|--------|
| Two Models | Logistic Regression + XGBoost logged in MLflow | ✅ |
| Champion | Best model registered in Model Registry | ✅ |
| Fairness Audit | DPR + EOR + PP across gender & socioeconomic groups → `dropout_fairness_audit` Delta table | ✅ |
| SHAP Global | Feature importance summary plot | ✅ |
| SHAP Per-Student | Top 3 risk factors for every flagged student | ✅ |
| Gold Table | `dropout_gold` — student_id, risk_score, tier, top 3 factors, action | ✅ |

**Next:** Open `dropout_dashboard.ipynb` for visual dashboard panels.